In [2]:
!pip -q install scikit-learn==1.7.2 pandas numpy joblib

In [3]:
import joblib, pandas as pd, numpy as np
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score


In [9]:
CSV = "/content/creditcard.csv"
MODEL = "/content/best_v1.joblib"

In [10]:
df = pd.read_csv(CSV)
y  = df["Class"].to_numpy()  # assumes standard dataset
features = ["Time"] + [f"V{i}" for i in range(1,29)] + ["Amount"]
X = df[features].to_numpy(dtype=float, copy=False)

In [12]:
model_dict = joblib.load(MODEL)
model = model_dict['pipeline']
scores = -model.decision_function(X)

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator RobustScaler from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator ExtraTreeRegressor from version 1.6.1 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator IsolationForest from version 1.6.1 when using version 1.7.2. This might lead to breaking code 

In [13]:

# Picking a threshold matching the dataset fraud rate
fraud_rate = y.mean()                  # ~0.0017 in this dataset
thr = np.percentile(scores, 100*(1-fraud_rate))
pred = (scores >= thr).astype(int)


In [14]:
p,r,f1,_ = precision_recall_fscore_support(y, pred, average="binary", zero_division=0)
auc = roc_auc_score(y, scores)

In [15]:
print(f"Precision={p:.3f}  Recall={r:.3f}  F1={f1:.3f}  ROC-AUC={auc:.3f}")

Precision=0.254  Recall=0.254  F1=0.254  ROC-AUC=0.946


In [16]:

import pandas as pd
pd.DataFrame([["best_v1 (IsolationForest)", p, r, f1, auc]],
             columns=["Model","Precision","Recall","F1","ROC_AUC"]).to_csv("/content/metrics.csv", index=False)
print("Saved /content/metrics.csv")

Saved /content/metrics.csv
